In [2]:
# Initial Dataset Inspection

import pandas as pd
from pathlib import Path

# Define project and dataset paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cpcb-aqi.csv"

print("Dataset exists:", DATA_PATH.exists())
print("Dataset size (MB):", round(DATA_PATH.stat().st_size / (1024 ** 2), 2))

# Load dataset
df = pd.read_csv(DATA_PATH, low_memory=False)

print("\nDataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset exists: True
Dataset size (MB): 78.83

Dataset loaded successfully!
Number of rows: 562878
Number of columns: 29


In [3]:
# Inspect column names and first five records

print("COLUMN NAMES:")
print(df.columns.tolist())

print("\nFIRST FIVE ROWS:")
display(df.head())

COLUMN NAMES:
['Station ID', 'State', 'City', 'Station Name', 'Date', '00:00:00', '01:00:00', '02:00:00', '03:00:00', '04:00:00', '05:00:00', '06:00:00', '07:00:00', '08:00:00', '09:00:00', '10:00:00', '11:00:00', '12:00:00', '13:00:00', '14:00:00', '15:00:00', '16:00:00', '17:00:00', '18:00:00', '19:00:00', '20:00:00', '21:00:00', '22:00:00', '23:00:00']

FIRST FIVE ROWS:


,Station ID,State,City,Station Name,Date,00:00:00,01:00:00,02:00:00,03:00:00,04:00:00,...,14:00:00,15:00:00,16:00:00,17:00:00,18:00:00,19:00:00,20:00:00,21:00:00,22:00:00,23:00:00
0,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-11,NaN,NaN,NaN,NaN,NaN,...,178.0,196.0,205.0,201.0,114.0,136.0,126.0,253.0,232.0,200.0
1,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-12,263.0,305.0,312.0,315.0,314.0,...,85.0,96.0,113.0,131.0,89.0,106.0,175.0,261.0,262.0,159.0
2,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-13,137.0,176.0,216.0,208.0,159.0,...,135.0,138.0,146.0,140.0,115.0,160.0,138.0,108.0,112.0,89.0
3,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-14,85.0,85.0,97.0,137.0,148.0,...,105.0,113.0,132.0,102.0,139.0,197.0,193.0,172.0,146.0,154.0
4,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-15,127.0,135.0,134.0,151.0,166.0,...,125.0,136.0,143.0,134.0,108.0,146.0,188.0,243.0,194.0,188.0


In [4]:
# Inspect data types and memory usage

print("DATASET INFORMATION:")
df.info()

print("\nDATA TYPES:")
print(df.dtypes)

DATASET INFORMATION:
<class 'pandas.DataFrame'>
RangeIndex: 562878 entries, 0 to 562877
Data columns (total 29 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Station ID    562878 non-null  str    
 1   State         562878 non-null  str    
 2   City          562878 non-null  str    
 3   Station Name  562878 non-null  str    
 4   Date          562878 non-null  str    
 5   00:00:00      522434 non-null  float64
 6   01:00:00      507667 non-null  float64
 7   02:00:00      514222 non-null  float64
 8   03:00:00      512977 non-null  float64
 9   04:00:00      510901 non-null  float64
 10  05:00:00      511948 non-null  float64
 11  06:00:00      509259 non-null  float64
 12  07:00:00      510917 non-null  float64
 13  08:00:00      509706 non-null  float64
 14  09:00:00      512619 non-null  float64
 15  10:00:00      517182 non-null  float64
 16  11:00:00      523332 non-null  float64
 17  12:00:00      526363 non-null  float64

In [5]:
# 1. DATE QUALITY AUDIT

date_parsed = pd.to_datetime(df["Date"], errors="coerce")

print("DATE AUDIT")
print("-" * 50)

print("Invalid dates:", date_parsed.isna().sum())
print("Earliest date:", date_parsed.min())
print("Latest date:", date_parsed.max())
print("Unique dates:", date_parsed.nunique())

print("\nYear-wise record count:")
print(date_parsed.dt.year.value_counts().sort_index())

DATE AUDIT
--------------------------------------------------
Invalid dates: 0
Earliest date: 2017-01-01 00:00:00
Latest date: 2023-12-31 00:00:00
Unique dates: 2541

Year-wise record count:
Date
2017     11441
2018     39928
2019     61586
2020     77125
2021     97214
2022    116567
2023    159017
Name: count, dtype: int64


In [6]:
# 2. MISSING VALUE AUDIT

hourly_cols = [
    col for col in df.columns
    if col not in ["Station ID", "State", "City", "Station Name", "Date"]
]

missing_count = df[hourly_cols].isna().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage.round(2)
})

print("MISSING VALUES IN HOURLY AQI COLUMNS")
display(missing_summary)

total_cells = len(df) * len(hourly_cols)
total_missing = df[hourly_cols].isna().sum().sum()

print("\nTotal hourly measurement cells:", total_cells)
print("Missing hourly measurements:", total_missing)
print(
    "Overall missing percentage:",
    round((total_missing / total_cells) * 100, 2), "%"
)

print(
    "\nRecords with all 24 hourly values missing:",
    df[hourly_cols].isna().all(axis=1).sum()
)

MISSING VALUES IN HOURLY AQI COLUMNS


,Missing_Count,Missing_Percentage
00:00:00,40444,7.19
01:00:00,55211,9.81
02:00:00,48656,8.64
03:00:00,49901,8.87
04:00:00,51977,9.23
05:00:00,50930,9.05
06:00:00,53619,9.53
07:00:00,51961,9.23
08:00:00,53172,9.45
09:00:00,50259,8.93



Total hourly measurement cells: 13509072
Missing hourly measurements: 1035531
Overall missing percentage: 7.67 %

Records with all 24 hourly values missing: 0


In [7]:
# 3. DUPLICATE AUDIT

print("DUPLICATE AUDIT")
print("-" * 50)

print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate Station ID + Date combinations:",
    df.duplicated(subset=["Station ID", "Date"]).sum()
)

print(
    "Duplicate Station Name + Date combinations:",
    df.duplicated(subset=["Station Name", "Date"]).sum()
)

DUPLICATE AUDIT
--------------------------------------------------
Exact duplicate rows: 0
Duplicate Station ID + Date combinations: 282846
Duplicate Station Name + Date combinations: 0


In [8]:
# 4. STATION AND GEOGRAPHIC COVERAGE

print("GEOGRAPHIC COVERAGE")
print("-" * 50)

print("Unique station IDs:", df["Station ID"].nunique())
print("Unique station names:", df["Station Name"].nunique())
print("Unique cities:", df["City"].nunique())
print("Unique states:", df["State"].nunique())

print("\nTop 15 states by record count:")
display(df["State"].value_counts().head(15).to_frame("Record_Count"))

print("\nTop 15 cities by record count:")
display(df["City"].value_counts().head(15).to_frame("Record_Count"))

print("\nStation ID mapping to multiple station names:")
station_name_mapping = (
    df.groupby("Station ID")["Station Name"]
    .nunique()
    .sort_values(ascending=False)
)

display(station_name_mapping.head(15).to_frame("Unique_Station_Names"))
print(
    "Station IDs mapped to multiple names:",
    (station_name_mapping > 1).sum()
)

GEOGRAPHIC COVERAGE
--------------------------------------------------
Unique station IDs: 248
Unique station names: 529
Unique cities: 248
Unique states: 29

Top 15 states by record count:


,Record_Count
State,
Haryana,82166
Delhi,81148
Uttar Pradesh,61495
Maharashtra,59770
Karnataka,46908
Bihar,31759
Madhya Pradesh,25586
Rajasthan,25200
West Bengal,22246



Top 15 cities by record count:


,Record_Count
City,
Delhi,81148
Yamuna Nagar,35066
Mumbai,33576
Bengaluru,21413
Hyderabad,17848
Chennai,12340
Kolkata,11743
Noida,11224
Patna,9249



Station ID mapping to multiple station names:


,Unique_Station_Names
Station ID,
site_5046,40
site_115,39
site_5964,38
site_5604,14
site_5678,14
site_5767,10
site_5363,9
site_5452,9
site_1405,7


Station IDs mapped to multiple names: 59


In [9]:
# 5. AQI VALUE AUDIT

print("AQI VALUE AUDIT")
print("-" * 50)

aqi_values = df[hourly_cols].stack()

print("Total non-missing AQI measurements:", len(aqi_values))
print("Minimum AQI:", aqi_values.min())
print("Maximum AQI:", aqi_values.max())

print("\nDescriptive statistics:")
display(aqi_values.describe().to_frame("AQI"))

print("\nMeasurements below zero:", (aqi_values < 0).sum())
print("Measurements above 500:", (aqi_values > 500).sum())
print("Measurements equal to zero:", (aqi_values == 0).sum())

AQI VALUE AUDIT
--------------------------------------------------
Total non-missing AQI measurements: 13509072
Minimum AQI: 1.0
Maximum AQI: 3797.0

Descriptive statistics:


,AQI
count,1.247354e+07
mean,1.364668e+02
std,1.075935e+02
min,1.000000e+00
25%,6.000000e+01
50%,1.000000e+02
75%,1.730000e+02
max,3.797000e+03



Measurements below zero: 0
Measurements above 500: 56
Measurements equal to zero: 0


In [10]:
# Corrected AQI measurement count

aqi_values_clean = df[hourly_cols].stack().dropna()

total_cells = len(df) * len(hourly_cols)
valid_measurements = len(aqi_values_clean)
missing_measurements = total_cells - valid_measurements

print("Total possible hourly cells:", total_cells)
print("Valid AQI measurements:", valid_measurements)
print("Missing AQI measurements:", missing_measurements)

print(
    "Missing percentage:",
    round(missing_measurements / total_cells * 100, 2),
    "%"
)

print("\nCorrected descriptive statistics:")
display(aqi_values_clean.describe())

Total possible hourly cells: 13509072
Valid AQI measurements: 12473541
Missing AQI measurements: 1035531
Missing percentage: 7.67 %

Corrected descriptive statistics:


count    1.247354e+07
mean     1.364668e+02
std      1.075935e+02
min      1.000000e+00
25%      6.000000e+01
50%      1.000000e+02
75%      1.730000e+02
max      3.797000e+03
dtype: float64

In [11]:
# STATION IDENTITY INVESTIGATION

station_identity = (
    df.groupby(["State", "City", "Station Name"])
    .agg(
        Unique_IDs=("Station ID", "nunique"),
        Record_Count=("Date", "size"),
        First_Date=("Date", "min"),
        Last_Date=("Date", "max")
    )
    .reset_index()
)

print("Unique station identity combinations:", len(station_identity))

print("\nStation identities associated with multiple IDs:")
display(
    station_identity[station_identity["Unique_IDs"] > 1]
    .sort_values("Unique_IDs", ascending=False)
    .head(20)
)

print("\nStation ID mapping to multiple locations/names:")

id_mapping = (
    df.groupby("Station ID")
    .agg(
        Unique_Station_Names=("Station Name", "nunique"),
        Unique_Cities=("City", "nunique"),
        Unique_States=("State", "nunique")
    )
    .sort_values("Unique_Station_Names", ascending=False)
)

display(id_mapping.head(20))

Unique station identity combinations: 529

Station identities associated with multiple IDs:


,State,City,Station Name,Unique_IDs,Record_Count,First_Date,Last_Date



Station ID mapping to multiple locations/names:


,Unique_Station_Names,Unique_Cities,Unique_States
Station ID,,,
site_5046,40,1,1
site_115,39,1,1
site_5964,38,1,1
site_5604,14,1,1
site_5678,14,1,1
site_5767,10,1,1
site_5363,9,1,1
site_5452,9,1,1
site_1405,7,1,1


In [12]:
# CHECK COMPOSITE STATION-DATE DUPLICATES

composite_key = ["State", "City", "Station Name", "Date"]

print(
    "Duplicate composite station-date records:",
    df.duplicated(subset=composite_key).sum()
)

Duplicate composite station-date records: 0


In [13]:
# INVESTIGATE AQI VALUES ABOVE 500

import numpy as np

aqi_array = df[hourly_cols].to_numpy()

row_indices, hour_indices = np.where(aqi_array > 500)

high_aqi_records = pd.DataFrame({
    "State": df.iloc[row_indices]["State"].to_numpy(),
    "City": df.iloc[row_indices]["City"].to_numpy(),
    "Station Name": df.iloc[row_indices]["Station Name"].to_numpy(),
    "Station ID": df.iloc[row_indices]["Station ID"].to_numpy(),
    "Date": df.iloc[row_indices]["Date"].to_numpy(),
    "Hour": np.array(hourly_cols)[hour_indices],
    "AQI": aqi_array[row_indices, hour_indices]
})

print("Number of readings above 500:", len(high_aqi_records))

display(
    high_aqi_records.sort_values("AQI", ascending=False)
)

Number of readings above 500: 56


,State,City,Station Name,Station ID,Date,Hour,AQI
51,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-17,11:00:00,3797.0
53,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,site_5111,2017-10-06,01:00:00,3562.0
52,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-18,11:00:00,3354.0
46,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-10,11:00:00,3303.0
47,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-11,11:00:00,2956.0
49,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-14,11:00:00,2902.0
48,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,site_153,2017-10-12,11:00:00,2845.0
54,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,site_5111,2017-10-17,01:00:00,2176.0
55,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,site_5111,2017-10-20,01:00:00,2164.0
4,Delhi,Delhi,NSIT Dwarka Delhi CPCB,site_115,2017-09-28,15:00:00,1182.0


In [14]:
# Summary of AQI readings above 500 by station

high_aqi_summary = (
    high_aqi_records
    .groupby(["State", "City", "Station Name"])
    .agg(
        High_Readings=("AQI", "count"),
        Minimum=("AQI", "min"),
        Maximum=("AQI", "max"),
        Average=("AQI", "mean"),
        First_Occurrence=("Date", "min"),
        Last_Occurrence=("Date", "max")
    )
    .reset_index()
    .sort_values("High_Readings", ascending=False)
)

display(high_aqi_summary)

,State,City,Station Name,High_Readings,Minimum,Maximum,Average,First_Occurrence,Last_Occurrence
3,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,40,538.0,900.0,860.450000,2017-09-20,2017-09-25
4,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,7,1051.0,3797.0,2886.857143,2017-10-10,2017-10-18
5,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,3,2164.0,3562.0,2634.000000,2017-10-06,2017-10-20
0,Andhra Pradesh,Vijayawada,PWD Grounds Vijayawada APPCB,3,526.0,852.0,656.666667,2017-10-04,2017-10-17
1,Delhi,Delhi,NSIT Dwarka Delhi CPCB,2,543.0,1182.0,862.500000,2017-09-28,2017-09-28
2,Delhi,Delhi,Sirifort Delhi CPCB,1,678.0,678.0,678.000000,2017-10-01,2017-10-01


In [15]:
# Frequency of each unusual AQI value

print("Frequency of values above 500:")
display(
    high_aqi_records["AQI"]
    .value_counts()
    .sort_values(ascending=False)
    .to_frame("Frequency")
)

print("\nReadings by year:")
display(
    high_aqi_records.assign(
        Year=pd.to_datetime(high_aqi_records["Date"]).dt.year
    )
    .groupby("Year")
    .size()
    .to_frame("High_Readings")
)

Frequency of values above 500:


,Frequency
AQI,
900.0,33
692.0,4
852.0,1
592.0,1
526.0,1
543.0,1
1182.0,1
678.0,1
798.0,1



Readings by year:


,High_Readings
Year,
2017,56


In [16]:
# Inspect the repeated 900 readings

ludhiana_900 = high_aqi_records[
    (high_aqi_records["City"] == "Ludhiana") &
    (high_aqi_records["AQI"] == 900)
].sort_values(["Date", "Hour"])

display(ludhiana_900)

,State,City,Station Name,Station ID,Date,Hour,AQI
7,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-20,22:00:00,900.0
8,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-20,23:00:00,900.0
9,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,00:00:00,900.0
10,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,01:00:00,900.0
11,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,02:00:00,900.0
12,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,03:00:00,900.0
13,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,04:00:00,900.0
14,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,05:00:00,900.0
15,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,06:00:00,900.0
18,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,site_253,2017-09-21,09:00:00,900.0


In [17]:
# Inspect all 24 hourly readings for records containing AQI > 500

suspicious_dates = high_aqi_records[
    ["State", "City", "Station Name", "Date"]
].drop_duplicates()

suspicious_rows = df.merge(
    suspicious_dates,
    on=["State", "City", "Station Name", "Date"],
    how="inner"
)

display(
    suspicious_rows[
        ["State", "City", "Station Name", "Date"] + hourly_cols
    ].sort_values(["City", "Date"])
)

,State,City,Station Name,Date,00:00:00,01:00:00,02:00:00,03:00:00,04:00:00,05:00:00,...,14:00:00,15:00:00,16:00:00,17:00:00,18:00:00,19:00:00,20:00:00,21:00:00,22:00:00,23:00:00
3,Delhi,Delhi,NSIT Dwarka Delhi CPCB,2017-09-28,295.0,322.0,313.0,307.0,314.0,333.0,...,543.0,1182.0,329.0,145.0,280.0,213.0,NaN,253.0,NaN,NaN
4,Delhi,Delhi,Sirifort Delhi CPCB,2017-10-01,312.0,327.0,320.0,328.0,470.0,678.0,...,155.0,149.0,113.0,115.0,136.0,151.0,153.0,114.0,150.0,225.0
16,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-06,101.0,3562.0,69.0,64.0,60.0,68.0,...,30.0,36.0,42.0,34.0,35.0,NaN,NaN,NaN,56.0,50.0
17,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-17,48.0,2176.0,58.0,39.0,34.0,34.0,...,48.0,52.0,49.0,51.0,42.0,44.0,53.0,44.0,60.0,56.0
18,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-20,348.0,2164.0,339.0,355.0,351.0,342.0,...,20.0,32.0,23.0,27.0,35.0,38.0,31.0,32.0,30.0,31.0
5,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-20,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,156.0,464.0,270.0,692.0,900.0,900.0
6,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-21,900.0,900.0,900.0,900.0,900.0,900.0,...,692.0,900.0,900.0,900.0,900.0,900.0,900.0,319.0,614.0,900.0
7,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-22,900.0,900.0,900.0,900.0,900.0,900.0,...,134.0,256.0,130.0,127.0,144.0,169.0,140.0,131.0,131.0,122.0
8,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-25,121.0,114.0,114.0,110.0,117.0,136.0,...,900.0,141.0,140.0,148.0,147.0,156.0,186.0,207.0,243.0,195.0
9,Uttar Pradesh,Noida,Sector 125 Noida UPPCB,2017-10-10,324.0,327.0,323.0,322.0,322.0,317.0,...,232.0,234.0,208.0,165.0,152.0,201.0,375.0,443.0,428.0,336.0


In [18]:
import pandas as pd
import numpy as np

hour_cols = [f"{h:02d}:00:00" for h in range(24)]

# Convert hourly values into a numeric matrix
hourly = df[hour_cols].apply(pd.to_numeric, errors="coerce")

# 1. Missingness by year
df_dates = pd.to_datetime(df["Date"], errors="coerce")

missing_by_year = pd.DataFrame({
    "Year": df_dates.dt.year,
    "Missing_Hours": hourly.isna().sum(axis=1),
    "Total_Hours": 24
})

year_summary = missing_by_year.groupby("Year").agg(
    Records=("Total_Hours", "size"),
    Missing_Hours=("Missing_Hours", "sum"),
    Total_Hours=("Total_Hours", "sum")
)

year_summary["Missing_Percentage"] = (
    year_summary["Missing_Hours"] /
    year_summary["Total_Hours"] * 100
)

display(year_summary.round(2))


# 2. Stations with highest missingness
station_data = df[["State", "City", "Station Name"]].copy()
station_data["Missing_Hours"] = hourly.isna().sum(axis=1)
station_data["Total_Hours"] = 24

station_summary = station_data.groupby(
    ["State", "City", "Station Name"]
).agg(
    Records=("Total_Hours", "size"),
    Missing_Hours=("Missing_Hours", "sum"),
    Total_Hours=("Total_Hours", "sum")
)

station_summary["Missing_Percentage"] = (
    station_summary["Missing_Hours"] /
    station_summary["Total_Hours"] * 100
)

display(
    station_summary.sort_values(
        "Missing_Percentage", ascending=False
    ).head(15).round(2)
)


# 3. Overall hourly missingness
missing_by_hour = hourly.isna().mean() * 100

display(
    missing_by_hour.rename("Missing_Percentage")
    .to_frame()
    .round(2)
)

,Records,Missing_Hours,Total_Hours,Missing_Percentage
Year,,,,
2017,11441,34447,274584,12.55
2018,39928,81098,958272,8.46
2019,61586,109260,1478064,7.39
2020,77125,123341,1851000,6.66
2021,97214,170976,2333136,7.33
2022,116567,224457,2797608,8.02
2023,159017,291952,3816408,7.65


Records  \
State             City            Station Name                                         
Arunachal Pradesh Naharlagun      Naharlagun Naharlagun APSPCB                   743   
Tamil Nadu        Chengalpattu    Crescent University Chengalpattu TNPCB         286   
Chhattisgarh      Bhilai          Civic Center Bhilai Bhilai Steel Plant         414   
Tamil Nadu        Tiruchirappalli St Joseph College Tiruchirappalli TNPCB         12   
                  Vellore         Vasanthapuram Vellore TNPCB                    399   
Maharashtra       Thane           Kasarvadavali Thane MPCB                       154   
                  Boisar          Khaira Boisar MPCB                             148   
Karnataka         Raichur         Mahatma Basaveswar Colony Kalaburgi KSPCB      272   
                  Udupi           Brahmagiri Udupi KSPCB                         812   
Jharkhand         Jorapokhar      Tata Stadium Jorapokhar JSPCB                 1674   
Tamil Nadu        Salem           Sona College of Technology Salem TNPCB         397   
Odisha            Rourkela        Sector-2 Rourkela OSPCB                        325   
Karnataka         Dharwad         Kalabhavan Dharwad KSPCB                       377   
Haryana           Gurugram        NISE Gwal Pahari Gurugram IMD                 1638   
Maharashtra       Pune            Alandi Pune IITM                               677   

                                                                             Missing_Hours  \
State             City            Station Name                                               
Arunachal Pradesh Naharlagun      Naharlagun Naharlagun APSPCB                        7795   
Tamil Nadu        Chengalpattu    Crescent University Chengalpattu TNPCB              2858   
Chhattisgarh      Bhilai          Civic Center Bhilai Bhilai Steel Plant              3633   
Tamil Nadu        Tiruchirappalli St Joseph College Tiruchirappalli TNPCB              101   
                  Vellore         Vasanthapuram Vellore TNPCB                         3272   
Maharashtra       Thane           Kasarvadavali Thane MPCB                            1239   
                  Boisar          Khaira Boisar MPCB                                  1142   
Karnataka         Raichur         Mahatma Basaveswar Colony Kalaburgi KSPCB           2080   
                  Udupi           Brahmagiri Udupi KSPCB                              6072   
Jharkhand         Jorapokhar      Tata Stadium Jorapokhar JSPCB                      12086   
Tamil Nadu        Salem           Sona College of Technology Salem TNPCB              2837   
Odisha            Rourkela        Sector-2 Rourkela OSPCB                             2270   
Karnataka         Dharwad         Kalabhavan Dharwad KSPCB                            2487   
Haryana           Gurugram        NISE Gwal Pahari Gurugram IMD                      10225   
Maharashtra       Pune            Alandi Pune IITM                                    3928   

                                                                             Total_Hours  \
State             City            Station Name                                             
Arunachal Pradesh Naharlagun      Naharlagun Naharlagun APSPCB                     17832   
Tamil Nadu        Chengalpattu    Crescent University Chengalpattu TNPCB            6864   
Chhattisgarh      Bhilai          Civic Center Bhilai Bhilai Steel Plant            9936   
Tamil Nadu        Tiruchirappalli St Joseph College Tiruchirappalli TNPCB            288   
                  Vellore         Vasanthapuram Vellore TNPCB                       9576   
Maharashtra       Thane           Kasarvadavali Thane MPCB                          3696   
                  Boisar          Khaira Boisar MPCB                                3552   
Karnataka         Raichur         Mahatma Basaveswar Colony Kalaburgi KSPCB         6528   
                  Udupi           Brahmagiri Udupi KSPCB     

,Missing_Percentage
00:00:00,7.19
01:00:00,9.81
02:00:00,8.64
03:00:00,8.87
04:00:00,9.23
05:00:00,9.05
06:00:00,9.53
07:00:00,9.23
08:00:00,9.45
09:00:00,8.93


In [20]:
# Parse dates
df_coverage = df.copy()
df_coverage["Date"] = pd.to_datetime(
    df_coverage["Date"], errors="coerce"
)

station_cols = ["State", "City", "Station Name"]

# 1. Annual monitoring network coverage
df_coverage["Year"] = df_coverage["Date"].dt.year

annual_coverage = df_coverage.groupby("Year").agg(
    Total_Records=("Date", "size"),
    Active_Stations=("Station Name", "nunique"),
    Active_Cities=("City", "nunique"),
    Active_States=("State", "nunique")
)

display(annual_coverage)


# 2. Station-level coverage across the full dataset
station_coverage = df_coverage.groupby(
    station_cols
).agg(
    First_Date=("Date", "min"),
    Last_Date=("Date", "max"),
    Recorded_Days=("Date", "nunique"),
    Total_Records=("Date", "size")
).reset_index()

station_coverage["Expected_Days"] = (
    station_coverage["Last_Date"] -
    station_coverage["First_Date"]
).dt.days + 1

station_coverage["Date_Coverage_Percentage"] = (
    station_coverage["Recorded_Days"] /
    station_coverage["Expected_Days"] * 100
)

display(
    station_coverage.sort_values(
        "Date_Coverage_Percentage"
    ).head(15)
)


# 3. Stations with the longest observation history
display(
    station_coverage.sort_values(
        "Recorded_Days", ascending=False
    ).head(15)
)


# 4. Number of stations appearing for the first time each year
first_year = (
    df_coverage.groupby(station_cols)["Year"]
    .min()
    .reset_index(name="First_Year")
)

new_stations = (
    first_year.groupby("First_Year")
    .size()
    .rename("New_Stations")
)

display(new_stations)

,Total_Records,Active_Stations,Active_Cities,Active_States
Year,,,,
2017,11441,83,49,17
2018,39928,134,69,17
2019,61586,201,102,20
2020,77125,255,122,22
2021,97214,321,151,25
2022,116567,396,188,29
2023,159017,519,247,29


,State,City,Station Name,First_Date,Last_Date,Recorded_Days,Total_Records,Expected_Days,Date_Coverage_Percentage
220,Karnataka,Karwar,KHB Colony Karwar KSPCB,2023-09-17,2023-12-31,14,14,106,13.207547
249,Madhya Pradesh,Gwalior,Phool Bagh Gwalior Mondelez Ind. Food,2020-01-01,2023-12-31,679,679,1461,46.475017
356,Meghalaya,Shillong,JN Stadium Shillong Meghalaya PCB,2022-04-06,2023-12-31,308,308,635,48.503937
55,Chhattisgarh,Bhilai,Civic Center Bhilai Bhilai Steel Plant,2021-11-16,2023-12-31,414,414,776,53.350515
339,Maharashtra,Pune,Bhosari Pune IITM,2020-11-07,2023-12-31,620,620,1150,53.913043
107,Gujarat,Ahmedabad,Gyaspur Ahmedabad IITM,2021-06-04,2023-12-31,517,517,941,54.941552
423,Tamil Nadu,Chengalpattu,Crescent University Chengalpattu TNPCB,2022-08-10,2023-12-29,286,286,507,56.410256
347,Maharashtra,Pune,Transport Nagar-Nigdi Pune IITM,2020-11-07,2023-12-31,652,652,1150,56.695652
343,Maharashtra,Pune,MIT-Kothrud Pune IITM,2021-01-12,2023-12-31,620,620,1084,57.195572
338,Maharashtra,Pune,Alandi Pune IITM,2020-11-14,2023-12-31,677,677,1143,59.230096


,State,City,Station Name,First_Date,Last_Date,Recorded_Days,Total_Records,Expected_Days,Date_Coverage_Percentage
100,Delhi,Delhi,Shadipur Delhi CPCB,2017-01-01,2023-12-31,2482,2482,2556,97.104851
88,Delhi,Delhi,NSIT Dwarka Delhi CPCB,2017-01-02,2023-12-31,2477,2477,2555,96.947162
86,Delhi,Delhi,Mandir Marg Delhi DPCC,2017-01-01,2023-12-31,2471,2471,2556,96.674491
75,Delhi,Delhi,DTU Delhi CPCB,2017-01-01,2023-12-31,2469,2469,2556,96.596244
79,Delhi,Delhi,IHBAS Dilshad Garden Delhi CPCB,2017-01-01,2023-12-31,2466,2466,2556,96.478873
101,Delhi,Delhi,Sirifort Delhi CPCB,2017-01-01,2023-12-31,2437,2437,2556,95.344288
10,Andhra Pradesh,Visakhapatnam,GVM Corporation Visakhapatnam APPCB,2017-01-01,2023-12-31,2433,2433,2556,95.187793
80,Delhi,Delhi,ITO Delhi CPCB,2017-01-01,2023-12-31,2426,2426,2556,94.913928
508,Uttar Pradesh,Varanasi,Ardhali Bazar Varanasi UPPCB,2017-01-01,2023-12-31,2415,2415,2556,94.483568
98,Delhi,Delhi,R K Puram Delhi DPCC,2017-01-01,2023-12-31,2398,2398,2556,93.818466


First_Year
2017     83
2018     51
2019     67
2020     56
2021     67
2022     76
2023    129
Name: New_Stations, dtype: int64